# SafeAnes UC04 — pilot từng bước
Bước đầu chạy CPU để kiểm tra dữ liệu, nhãn và baseline. Chưa có kết quả T4/deep learning.
Nguồn: [VitalDB API](https://vitaldb.net/docs/?documentId=API%2FWeb_API_OpenDataset.md), [VitalDB paper](https://doi.org/10.1038/s41597-022-01411-5), [selection bias](https://pubmed.ncbi.nlm.nih.gov/40404499/), [calibration](https://scikit-learn.org/stable/modules/calibration.html). Xem `docs/SOURCES.md` và `docs/PROTOCOL.md` trong source.
Kaggle: upload source vào Input, sửa PROJECT_ROOT bên dưới; bật Internet cho bước tải. Pilot chỉ dùng global train; final test không được mở.

In [ ]:
from pathlib import Path
import sys, json, importlib.util
import pandas as pd
ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/input/safeanes-source') if ON_KAGGLE else Path.cwd()  # sửa theo Input thực tế
assert (PROJECT_ROOT / 'src/safeanes').exists(), 'Sửa PROJECT_ROOT thành thư mục chứa src/safeanes'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
WORK_ROOT = Path('/kaggle/working/safeanes') if ON_KAGGLE else PROJECT_ROOT
RUN_NAME = 'pilot_v1'  # đổi tên khi thay protocol hoặc chạy thí nghiệm mới
RAW_ROOT = WORK_ROOT / 'data/vitaldb'
DATASET = WORK_ROOT / 'data' / RUN_NAME
OUTPUT = WORK_ROOT / 'artifacts' / RUN_NAME
from safeanes.config import Protocol
from safeanes.data import fetch_pilot
from safeanes.dataset import build_pilot
from safeanes.experiment import run_pilot
protocol = Protocol()
print('Protocol:', protocol.digest())
print('LightGBM available:', importlib.util.find_spec('lightgbm') is not None)

## 1. Tải và kiểm kê
Chỉ tải numeric của 60 ca train. Cache giữ URL/hash/thời điểm; lệnh tải chạy lại được khi gián đoạn. Không tải toàn bộ waveform.

In [ ]:
manifest = fetch_pilot(RAW_ROOT, limit=60, workers=4, protocol=protocol)
assert manifest['split'].eq('train').all()
display(manifest[['caseid', 'subjectid', 'age', 'opstart', 'opend', 'split']].head())
print((RAW_ROOT / 'fetch_report.json').read_text())

## 2. Dựng dataset và audit nhãn
Không ghi đè dataset cũ. Chỉ chạy một lần cho mỗi RUN_NAME. Nhãn -1 nghĩa thiếu follow-up, không phải không có biến cố. Các ca của một người cùng split.

In [ ]:
print(build_pilot(RAW_ROOT, DATASET, protocol))
quality = pd.read_csv(DATASET / 'quality.csv')
display(quality.describe())
windows = pd.read_csv(DATASET / 'windows.csv.gz')
display(windows[['eligible', 'y_300', 'y_600']].value_counts().rename('windows').reset_index())
print('Label unknown rate:', {h: float(windows[f'y_{h}'].eq(-1).mean()) for h in [300, 600]})

## 3. Baseline + calibration + cảnh báo
Model fit, calibration, chọn threshold và pilot_test dùng bệnh nhân riêng. Đây đều là tập con global train; mọi metric chỉ thăm dò. Bootstrap theo bệnh nhân, không theo window.

In [ ]:
MODELS = ['map', 'logistic', 'hist_gradient']
# Có thể thêm 'lightgbm' khi dependency đã có; không thay tên hist_gradient.
report = run_pilot(DATASET, OUTPUT, models=MODELS, repeats=200)
assert not report['errors'], report['errors']
rows = []
for name, result in report['models'].items():
    m = result['pilot_test']
    rows.append({'model': name, **{k: m[k] for k in ['auroc', 'average_precision', 'event_sensitivity', 'alarm_ppv', 'false_alarms_per_hour', 'events_eligible', 'events_detected']}, 'point_targets_met': result['gates']['all_point_targets_met']})
display(pd.DataFrame(rows))
print('Scope:', report['scope'])
print('Detailed report:', OUTPUT / 'report.json')

## 4. Phát lại một ca để kiểm tra lỗi
Chọn ca có event từ kết quả, hiển thị MAP/risk và alarm. Biểu đồ này để audit hồi cứu; không phải giao diện theo dõi bệnh nhân thật. Matplotlib là dependency có sẵn trên Kaggle.

In [ ]:
import matplotlib.pyplot as plt
MODEL_KEY = 'hist_gradient_300'
pred = pd.read_csv(OUTPUT / f'{MODEL_KEY}_predictions.csv.gz')
events = pd.read_csv(DATASET / 'events.csv')
available = events[events.caseid.isin(pred.caseid)]
caseid = int(available.caseid.iloc[0]) if len(available) else int(pred.caseid.iloc[0])
case = pred[pred.caseid.eq(caseid)].sort_values('time')
alarms = json.loads((OUTPUT / f'{MODEL_KEY}_alarms.json').read_text())
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 6))
axes[0].plot(case.time / 60, case.map_current, label='MAP sampled at decisions')
axes[0].axhline(65, color='red', linestyle='--')
axes[0].set_ylabel('mmHg')
axes[1].plot(case.time / 60, case.probability, label='Calibrated risk')
axes[1].axhline(report['models'][MODEL_KEY]['pilot_test']['threshold'], color='orange', linestyle='--')
for event in events[events.caseid.eq(caseid)].itertuples():
    for ax in axes: ax.axvspan(event.onset / 60, event.end / 60, alpha=.15, color='red')
for alarm in alarms:
    if alarm['caseid'] == caseid: axes[1].axvline(alarm['time'] / 60, color='purple', alpha=.7)
axes[1].set_xlabel('Minutes from casestart')
axes[1].set_ylabel('Probability')
axes[0].set_title(f'Exploratory replay: case {caseid}')
for ax in axes: ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'case_replay.png', dpi=150)
plt.show()

## Bước tiếp theo
Đọc quality.csv và false/missed alerts trước khi tăng model complexity. Mở rộng train-pool pilot theo manifest cố định; audit eligibility/censoring và exposure. Sau đó triển khai TCN trên cùng protocol, đo T4/VRAM thật, rồi so với baseline.
AUROC cao không tự chứng minh cảnh báo có ích. Chỉ báo đạt khi mọi tiêu chí đã khóa được kiểm chứng trên final test độc lập; pilot chưa đủ cho kết luận đó.